# Ablation Row 5 — + Detection-level threshold tuning (inference only)

| | Value |
|---|---|
| Features | full (same as row 4) |
| PCA | 4677 components |
| Scales | (0.75, 1.0, 1.5) + cross-scale NMS |
| HNM | 4 rounds (loaded from row 4 pkls) |
| Threshold | **tuned** — sweep against IoU≥0.5 P/R/F1 on validate images |

Delta vs row 4: replace the default `cfg.detection_threshold = 1.5` with the
value that maximises detection-level F1 on the validate set.

**No retraining** — loads `microglia-artifacts-row4/`'s pkls, runs
`tune_detection_threshold` (cached-score sweep, ~30 min), then re-evaluates on
the held-out test images at the tuned threshold. This is the headline pipeline
number for the presentation.

## 1  Imports + Config + load row 4 model

In [ ]:
import os
import joblib
import numpy as np
import matplotlib.image as mpimg

from pipeline import (
    Config,
    extract_raw_features, fit_pipeline,
    tune_svm, train_svm, evaluate_classifier,
    process_image, multi_scale_detect, non_max_suppression,
    load_gt_boxes, evaluate_detections, plot_gt_vs_pred,
    save_hard_negatives, tune_detection_threshold,
    ensure_test_patches, patch_level_test_eval, detection_level_test_eval,
    extract_features, list_images,
)

%matplotlib inline

In [ ]:
cfg = Config(
    artifact_dir='./microglia-artifacts-row4',
    feature_mode='full',
    scale_factors=(0.75, 1.0, 1.5),
    pca_n_components=4677,
)
print(cfg)

svm_clf = joblib.load(cfg.svm_clf_path)
scaler  = joblib.load(cfg.scaler_path)
pca     = joblib.load(cfg.pca_path)
print(f"Loaded row 4 model from {cfg.artifact_dir}/")

# Reload the final HNM-trained X_val / y_val from features_cache (raw) + transform
# so tune_detection_threshold has the validate-set scores it needs.
import os.path as _p
cache = np.load(cfg.features_cache)
X_val_raw, y_val = cache['X_val_raw'], cache['y_val_raw']
X_val = scaler.transform(X_val_raw)
if pca is not None:
    X_val = pca.transform(X_val)

source_paths = list_images(cfg.source_images)
from sklearn.model_selection import train_test_split as _split
_, val_paths = _split(source_paths, test_size=cfg.val_size, random_state=cfg.random_state)
print(f"val_paths: {len(val_paths)} images for threshold sweep")

## 2  Detection-level threshold tuning on validate set

In [ ]:
best_t = tune_detection_threshold(
    val_paths, svm_clf, scaler, cfg, X_val, y_val, pca=pca, n_steps=15,
)
print(f"\ncfg.detection_threshold set to: {cfg.detection_threshold:.4f}")

## 3  Held-out test evaluation at tuned threshold

In [ ]:
ensure_test_patches(cfg)

print("── Patch-level test (unchanged from row 4) ──")
patch_level_test_eval(svm_clf, scaler, pca, cfg)

print("\n── Detection-level test (multi-scale, TUNED t=%.2f) ──" % cfg.detection_threshold)
row5_metrics = detection_level_test_eval(svm_clf, scaler, pca, cfg, show_plots=True)